# ResNet-18 grader and referral experiments

The notebook records the experiment configuration, analysis procedure and saved outputs for this study.


## 1. Frozen configuration and output isolation


In [ ]:
from __future__ import annotations
import os, re, json, hashlib, time, math, random, copy, platform
from pathlib import Path
from urllib.parse import urlparse
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, roc_auc_score
from tqdm.auto import tqdm
from IPython.display import display

REVISION='V51-SPATIAL-LOCK-R2-2026-10-22'
V4=Path(os.environ.get('DATEPALM_V4_ROOT','external_data')).expanduser().resolve()
STAGE1=Path(os.environ.get('DATEPALM_PILOT_OUTPUT','external_data')).expanduser().resolve()
OUT=Path(os.environ.get('DATEPALM_V51_OUT',str(STAGE1/'v51_locked_spatial'))).expanduser().resolve()
CAL_BENCH=V4/'calibration/benchmark'
TEST_BENCH=V4/'results/fixed_benchmark'
SEEDS=(11,23,37)
BUDGETS=(0.05,0.10,0.15)
PRIMARY_BUDGET=0.10
TRAIN_BATCH=32
INFER_BATCH=int(os.environ.get('DATEPALM_V51_INFER_BATCH','48'))
EPOCHS=12
PATIENCE=4
LR=0.0002
WEIGHT_DECAY=0.001
PATCH_DIM=64
TOP_PATCHES=5
COV_SHRINKAGE=0.10
BOOT_REPS=2000
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert 1 <= INFER_BATCH <= 128
if DEVICE.type!='cuda' and os.environ.get('DATEPALM_V51_ALLOW_CPU','0')!='1':
    raise RuntimeError('CUDA is not currently available. This notebook stops before training. '
                       'Use a free approved GPU session, or set DATEPALM_V51_ALLOW_CPU=1 explicitly (slow).')
assert (V4/'data/source_split.csv').is_file() and (STAGE1/'verified_v4_image_path_map.csv').is_file()
assert OUT not in [V4,STAGE1] and V4 not in OUT.parents and STAGE1 in OUT.parents
assert not (OUT/'RESULTS_COMPLETE.json').exists(), ('Completed results already exist at '+str(OUT)+'. '
    'Do not rerun/overwrite a completed locked experiment. Select a new output folder only for a documented rerun.')
OUT.mkdir(parents=True,exist_ok=True)
print('Run revision:',REVISION)
print('V4 READ ONLY:',V4)
print('Stage 1 image map READ ONLY:',STAGE1)
print('New output folder:',OUT)
print('Torch:',torch.__version__,'CUDA:',torch.cuda.is_available(),'Device:',DEVICE)
if DEVICE.type=='cuda':print('Available GPU:',torch.cuda.get_device_name(0))
print('Seeds:',SEEDS,'Max epochs:',EPOCHS,'Primary referral budget:',PRIMARY_BUDGET)

## 2. Full input and identity verification — before model training


In [ ]:
def must(condition,message):
    if not bool(condition):raise RuntimeError(message)

def read_text_csv(path):
    must(path.is_file(),f'Required frozen file missing: {path}')
    return pd.read_csv(path,dtype=str,keep_default_na=False)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def atomic_json(path,record):
    path=Path(path);temp=path.with_name(path.name+'.partial')
    with open(temp,'w') as f:json.dump(record,f,indent=2,sort_keys=True,default=str)
    os.replace(temp,path)

def fingerprint_records(frame, cols):
    h=hashlib.sha256()
    for record in frame[list(cols)].itertuples(index=False,name=None):
        h.update(json.dumps([str(x) for x in record],separators=(',',':')).encode('utf8'))
        h.update(b'\n')
    return h.hexdigest()

source=read_text_csv(V4/'data/source_split.csv')
source=source.loc[source.eligibility.eq('eligible')].copy()
must(len(source)==5828 and source.source_id.is_unique,'Frozen source count or source IDs changed')
must(source.split.value_counts().to_dict()=={'train':3603,'development':835,'calibration':556,'locked_test':834},
     'Frozen split sizes changed')
must(source.groupby('source_group').split.nunique().max()==1,'Source groups cross split boundaries')
label_order=['Small','Medium','Large']
label_ids={k:i for i,k in enumerate(label_order)}
must(set(source.native_size_label)==set(label_order),'Unknown size label')
image_map=read_text_csv(STAGE1/'verified_v4_image_path_map.csv')
must(len(image_map)==5828 and image_map.source_id.is_unique,'Verified original-image map invalid')
src=source.merge(image_map[['source_id','file_sha256','relative_path','verified_image_path']],
                 on='source_id',how='left',validate='one_to_one',suffixes=('','_mapped'),indicator=True)
must(src._merge.eq('both').all(),'Incomplete original-image identity map')
for col in ['file_sha256','relative_path']:
    must(src[col].eq(src[col+'_mapped']).all(),f'Image map {col} does not match V4 manifest')
src=src.drop(columns=['_merge','file_sha256_mapped','relative_path_mapped'])
must(all(Path(x).is_file() for x in src.verified_image_path),'Original photo missing')

def read_benchmark(folder,score_csv,split,nrows):
    b=read_text_csv(folder/'manifest.csv')
    s=read_text_csv(score_csv)
    must(len(b)==len(s)==nrows,f'{split}: unexpected row count')
    must(b.sample_id.is_unique and s.sample_id.is_unique,f'{split}: duplicate sample IDs')
    must(set(b.sample_id)==set(s.sample_id),f'{split}: benchmark/score sample IDs differ')
    bidx=b.set_index('sample_id').sort_index()
    sidx=s.set_index('sample_id').sort_index()
    for col in ['source_id','source_group','source_size','source_variety','kind',
                'family','severity','label','image_sha256']:
        must(bidx[col].equals(sidx[col]),f'{split}: benchmark/score mismatch in {col}')
    meta=src.loc[src.split.eq(split),['source_id','source_group','native_size_label','variety']].copy()
    joined=s.merge(meta,on='source_id',how='left',validate='many_to_one',suffixes=('','_source'),indicator=True)
    must(joined._merge.eq('both').all(),f'{split}: source outside frozen split')
    for first,second in [('source_group','source_group_source'),('source_size','native_size_label'),
                         ('source_variety','variety')]:
        must(joined[first].eq(joined[second]).all(),f'{split}: metadata differs in {first}')
    must(set(joined.source_id)==set(meta.source_id),f'{split}: missing benchmark sources')
    joined=joined.drop(columns=['_merge','source_group_source'])
    joined['image_path']=[str(folder/'images'/f'{sid}.png') for sid in joined.sample_id]
    must(all(Path(p).is_file() for p in joined.image_path),f'{split}: missing benchmark image PNG')
    must(joined.loc[joined.kind.eq('original'),'label'].eq('0').all(),f'{split}: original label invalid')
    must(joined.loc[joined.kind.str.startswith('benign'),'label'].eq('0').all(),f'{split}: benign label invalid')
    must(joined.groupby('source_id').size().eq(3 if split=='calibration' else 15).all(),
         f'{split}: wrong number of variants per source')
    return joined.reset_index(drop=True)

cal=read_benchmark(CAL_BENCH,V4/'calibration/scores/C11_1049.csv','calibration',1668)
test=read_benchmark(TEST_BENCH,V4/'results/C11_1049.csv','locked_test',12510)
must(test.kind.eq('original').sum()==834 and test.kind.str.startswith('benign').sum()==1668,
     'Test original/benign counts changed')
must(test.label.eq('1').sum()==10008,'Test synthetic edit count changed')
must(set(test.loc[test.label.eq('1'),'family'])=={'HO_T1','HO_T2','HO_T3','HO_T4'},'Edit families changed')
must(set(test.loc[test.label.eq('1'),'severity'])=={'low','medium','high'},'Edit severities changed')

input_manifest_fingerprint=fingerprint_records(src,['source_id','file_sha256','split','native_size_label','source_group'])
cal_fingerprint=fingerprint_records(cal,['sample_id','image_sha256','source_id','kind'])
test_fingerprint=fingerprint_records(test,['sample_id','image_sha256','source_id','kind'])
PROVENANCE={'revision':REVISION,'source_manifest_sha256':input_manifest_fingerprint,
            'calibration_benchmark_sha256':cal_fingerprint,
            'test_benchmark_sha256':test_fingerprint,'seeds':SEEDS,'budgets':BUDGETS,
            'classifier':{'architecture':'ResNet18','trainable':'layer4+fc','epochs_max':EPOCHS,
                          'patience':PATIENCE,'lr':LR,'weight_decay':WEIGHT_DECAY,'batch':TRAIN_BATCH},
            'anomaly':{'spatial_stage':'layer4','patch_size_grid':[7,7],
                       'random_projection_dim':PATCH_DIM,'top_patches':TOP_PATCHES,
                       'covariance_shrinkage':COV_SHRINKAGE}}
if (OUT/'provenance.json').exists():
    old=json.load(open(OUT/'provenance.json'))
    must(old==json.loads(json.dumps(PROVENANCE)), 'An incomplete prior run used different inputs or settings. STOP.')
else:atomic_json(OUT/'provenance.json',PROVENANCE)

for split_name,frame,path_col,sha_col in [
    ('original',src,'verified_image_path','file_sha256'),
    ('calibration',cal,'image_path','image_sha256'),
    ('locked_test',test,'image_path','image_sha256')]:
    mismatches=[]
    for path,digest in tqdm(frame[[path_col,sha_col]].itertuples(index=False,name=None),total=len(frame),
                            desc=f'Full SHA-256 {split_name}'):
        if sha256_file(path)!=digest:
            mismatches.append(path)
            if len(mismatches)>=5:break
    must(not mismatches,f'{split_name}: original image byte identities disagree: {mismatches}')
    print(f'PASS: SHA-256 verified every {split_name} file ({len(frame):,})')
print('PASS: immutable identity and split check for all 20,006 inputs (5,828 originals + 14,178 PNGs).')

## 3. Pretrained encoder and exact V4 image preparation


In [ ]:
from PIL import Image as PILImage

WEIGHTS=models.ResNet18_Weights.IMAGENET1K_V1
checkpoint_name=Path(urlparse(WEIGHTS.url).path).name
weight_file=Path(torch.hub.get_dir())/'checkpoints'/checkpoint_name
must(weight_file.is_file(),f'ImageNet pretrained weights not cached locally: {weight_file}. '
     'Use the same verified pretrained checkpoint as Stage 1; do not silently substitute random weights.')
weights_sha256=sha256_file(weight_file)
print('Pretrained weights:',weight_file,'SHA-256:',weights_sha256)
if (OUT/'weights_provenance.json').exists():
    old=json.load(open(OUT/'weights_provenance.json'))
    must(old['weights_sha256']==weights_sha256,'Pretrained weights changed since previous run')
else:atomic_json(OUT/'weights_provenance.json',{'weights_file':str(weight_file),
         'weights_sha256':weights_sha256,'torch':torch.__version__,'torchvision_weights':str(WEIGHTS)})
# Read exact checkpoint (no network access, no auto-download).
base=models.resnet18(weights=None)
base.load_state_dict(torch.load(weight_file,map_location='cpu',weights_only=True),strict=True)
base.eval()

PREPROCESS=transforms.Compose([
    transforms.Resize((224,224),interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
])

def v4_canonical_image(photo):
    with PILImage.open(photo) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        im.thumbnail((256,256),PILImage.Resampling.BICUBIC)
        canvas=PILImage.new('RGB',(256,256),(128,128,128))
        canvas.paste(im,((256-im.width)//2,(256-im.height)//2))
        return canvas

raw_lookup=src.set_index('source_id').verified_image_path.to_dict()
for partition,frame,folder in [('calibration',cal,CAL_BENCH),('locked_test',test,TEST_BENCH)]:
    original=frame.loc[frame.kind.eq('original')]
    for record in tqdm(original.itertuples(index=False),total=len(original),desc=f'Pixel match {partition}'):
        expected=np.asarray(v4_canonical_image(raw_lookup[record.source_id]))
        with PILImage.open(folder/'images'/f'{record.sample_id}.png') as im:
            actual=np.asarray(im.convert('RGB'))
        must(np.array_equal(expected,actual),f'Frozen V4 benchmark/preprocessing mismatch: {record.sample_id}')
print('PASS: All 1,390 clean V4 benchmark PNGs exactly match original-photo V4 preprocessing.')

class Images(Dataset):
    def __init__(self,frame,canonical):
        self.records=frame[['sample_id','image_path']].to_dict('records') if not canonical else [
            {'sample_id':r.source_id,'image_path':r.verified_image_path} for r in frame.itertuples(index=False)]
        self.canonical=canonical
    def __len__(self):return len(self.records)
    def __getitem__(self,index):
        record=self.records[index]
        if self.canonical:
            im=v4_canonical_image(record['image_path'])
        else:
            with PILImage.open(record['image_path']) as image:im=image.convert('RGB')
        return PREPROCESS(im),str(record['sample_id'])

class StemToLayer3(nn.Module):
    def __init__(self,model):
        super().__init__()
        self.conv1=model.conv1;self.bn1=model.bn1;self.relu=model.relu
        self.maxpool=model.maxpool;self.layer1=model.layer1;self.layer2=model.layer2;self.layer3=model.layer3
    def forward(self,x):
        return self.layer3(self.layer2(self.layer1(self.maxpool(self.relu(self.bn1(self.conv1(x)))))))

stem=StemToLayer3(base).eval().to(DEVICE)
for p in stem.parameters():p.requires_grad_(False)
class SizeHead(nn.Module):
    def __init__(self,original_layer4):
        super().__init__();self.layer4=copy.deepcopy(original_layer4)
        self.fc=nn.Linear(512,3)
    def spatial(self,feat):return self.layer4(feat)
    def forward(self,feat):
        spatial=self.spatial(feat)
        return self.fc(spatial.mean(dim=(-2,-1)))

# Dry-run checks before writing >1 GB of feature caches.
dryset=Images(src.loc[src.split.eq('train')].iloc[:4],canonical=True)
example=torch.stack([dryset[i][0] for i in range(4)]).to(DEVICE)
with torch.no_grad():
    intermediate=stem(example)
must(intermediate.shape==(4,256,14,14),f'Unexpected frozen stem shape: {intermediate.shape}')
probe=SizeHead(base.layer4).to(DEVICE)
logits=probe(intermediate)
must(logits.shape==(4,3),'Grader dry-run shape failed')
loss=nn.functional.cross_entropy(logits,torch.tensor([0,1,2,0],device=DEVICE))
loss.backward()
must(all(p.grad is not None and torch.isfinite(p.grad).all() for p in probe.parameters()),
     'Trainable layer4+fc failed its first backward pass')
probe.zero_grad(set_to_none=True)
del probe, example, intermediate, logits, loss
if DEVICE.type=='cuda':torch.cuda.empty_cache()
print('PASS: actual JPEG -> exact V4-prepared pixels -> frozen stem -> spatial layer4 -> size loss backward.')

## 4. Extract the frozen ImageNet stem features **once**


In [ ]:
import shutil
FREE_GB=shutil.disk_usage(OUT).free/1e9
print(f'Output filesystem free space: {FREE_GB:.2f} GB')
must(FREE_GB>4.0,'Less than 4 GB free disk space; cannot safely build the fixed spatial feature cache')

splits={'train':src.loc[src.split.eq('train')].reset_index(drop=True),
        'development':src.loc[src.split.eq('development')].reset_index(drop=True),
        'calibration':cal.reset_index(drop=True),
        'locked_test':test.reset_index(drop=True)}
CACHED_SHAPE=(256,14,14)

@torch.inference_mode()
def intermediate_cache(part,frame,canonical):
    target=OUT/f'fixed_layer3_{part}.npy';meta=OUT/f'fixed_layer3_{part}.json'
    rows=frame[['source_id','file_sha256']].rename(columns={'source_id':'id','file_sha256':'sha'}) \
        if canonical else frame[['sample_id','image_sha256']].rename(columns={'sample_id':'id','image_sha256':'sha'})
    digest=fingerprint_records(rows,['id','sha'])
    expected={'revision':REVISION,'split':part,'n':len(frame),'shape':[len(frame),*CACHED_SHAPE],
              'dtype':'float16','row_sha256':digest,'pretrained_weights_sha256':weights_sha256,
              'preprocessing':'v4_256_letterbox_then_resize_224_imagenet_norm'}
    if target.exists() or meta.exists():
        must(target.exists() and meta.exists(),f'Partial cache left for {part}; STOP, inspect {target}')
        must(json.load(open(meta))==expected,f'{part} feature-cache settings or IDs changed')
        arr=np.load(target,mmap_mode='r',allow_pickle=False)
        must(arr.shape==tuple(expected['shape']) and arr.dtype==np.float16,
             f'{part} cache shape/dtype changed')
        for j in sorted(set([0,len(frame)//3,len(frame)//2,len(frame)-1])):
            must(np.isfinite(arr[j]).all(),f'{part} cached features contain nonfinite values')
        print(f'REUSED verified frozen stem cache {part}: {arr.shape}')
        return arr
    temp=OUT/f'fixed_layer3_{part}.partial.npy'
    if temp.exists():
        print('Discarding incomplete temporary cache from interrupted run:',temp)
        temp.unlink()
    arr=np.lib.format.open_memmap(temp,mode='w+',dtype=np.float16,
                                  shape=(len(frame),*CACHED_SHAPE))
    ds=Images(frame,canonical=canonical)
    loader=DataLoader(ds,batch_size=INFER_BATCH,shuffle=False,num_workers=0,pin_memory=False)
    pos=0
    expected_ids=rows.id.tolist()
    for images,ids in tqdm(loader,desc=f'Frozen stem {part}'):
        must(list(ids)==expected_ids[pos:pos+len(ids)],f'{part}: DataLoader image order changed')
        vals=stem(images.to(DEVICE)).detach().cpu().numpy().astype(np.float16)
        must(np.isfinite(vals).all(),f'{part}: intermediate features contain NaN/Inf')
        arr[pos:pos+len(ids)]=vals
        pos+=len(ids)
    must(pos==len(frame),f'{part}: incomplete extraction')
    arr.flush();del arr
    os.replace(temp,target)
    atomic_json(meta,expected)
    arr=np.load(target,mmap_mode='r',allow_pickle=False)
    print(f'SAVED verified frozen stem {part}: {arr.shape}')
    return arr

fixed={part:intermediate_cache(part,frame,part in ('train','development'))
       for part,frame in splits.items()}
print('PASS: All frozen intermediate feature caches ready; V4 models have not been touched.')

## 5. Three size-trained copies of `layer4`; development-only checkpoint selection


In [ ]:
def random_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    if hasattr(torch.backends.cuda,'matmul'):
        torch.backends.cuda.matmul.allow_tf32=False

class CachedStemDataset(Dataset):
    def __init__(self,arr,labels):
        self.arr=arr;self.labels=np.asarray(labels,dtype=np.int64)
    def __len__(self):return len(self.labels)
    def __getitem__(self,index):
        # Return a writable copy; do not modify disk-backed caches.
        data=np.array(self.arr[index],dtype=np.float32,copy=True)
        return torch.from_numpy(data),int(self.labels[index])

Y_TRAIN=np.asarray([label_ids[s] for s in splits['train'].native_size_label],np.int64)
Y_DEV=np.asarray([label_ids[s] for s in splits['development'].native_size_label],np.int64)

@torch.inference_mode()
def predict_logits(net,arr,batch=INFER_BATCH):
    net.eval();outputs=[]
    for start in range(0,len(arr),batch):
        x=torch.from_numpy(np.asarray(arr[start:start+batch],dtype=np.float32)).to(DEVICE)
        outputs.append(net(x).detach().cpu().numpy().astype(np.float32))
    return np.concatenate(outputs,axis=0)

def train_or_load(seed):
    fn=OUT/f'size_layer4_seed{seed}.pt'
    random_seed(seed)
    model=SizeHead(base.layer4).to(DEVICE)
    config={'revision':REVISION,'seed':seed,'source_manifest_sha256':input_manifest_fingerprint,
            'weights_sha256':weights_sha256,'epochs_max':EPOCHS,'patience':PATIENCE,
            'train_batch':TRAIN_BATCH,'lr':LR,'weight_decay':WEIGHT_DECAY}
    if fn.exists():
        record=torch.load(fn,map_location='cpu',weights_only=True)
        must(record['config']==config,f'Checkpoint configuration mismatch: {fn}')
        model.load_state_dict(record['model_state'],strict=True)
        print(f'REUSED development-selected seed {seed}: epoch {record["best_epoch"]}, '
              f'dev macro F1 {record["best_macro_f1"]:.4f}')
        model.eval();return model,record
    optim=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    history=[];best_score=-np.inf;best_state=None;best_epoch=0;stale=0
    generator=torch.Generator().manual_seed(seed)
    loader=DataLoader(CachedStemDataset(fixed['train'],Y_TRAIN),batch_size=TRAIN_BATCH,
                      shuffle=True,generator=generator,num_workers=0,pin_memory=False)
    for epoch in range(1,EPOCHS+1):
        model.train();running=[]
        for feat,y in tqdm(loader,desc=f'Seed {seed} epoch {epoch}',leave=False):
            feat=feat.to(DEVICE);y=y.to(DEVICE)
            out=model(feat)
            loss=nn.functional.cross_entropy(out,y)
            must(torch.isfinite(loss),f'Seed {seed}: nonfinite training loss')
            optim.zero_grad(set_to_none=True);loss.backward();optim.step()
            running.append(float(loss.detach().cpu()))
        dev_pred=predict_logits(model,fixed['development']).argmax(1)
        macro=float(f1_score(Y_DEV,dev_pred,average='macro',labels=[0,1,2],zero_division=0))
        acc=float(accuracy_score(Y_DEV,dev_pred))
        history.append({'seed':seed,'epoch':epoch,'training_loss':float(np.mean(running)),
                        'development_macro_f1':macro,'development_accuracy':acc})
        print(f'Seed {seed} epoch {epoch}: train loss {np.mean(running):.4f}; '
              f'dev macro-F1 {macro:.4f}; dev accuracy {acc:.4f}')
        if macro>best_score+1e-6:
            best_score=macro;best_epoch=epoch;stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:stale+=1
        if stale>=PATIENCE:break
    must(best_state is not None,'No valid size-classifier checkpoint')
    model.load_state_dict(best_state);model.eval()
    record={'config':config,'model_state':best_state,'history':history,
            'best_epoch':best_epoch,'best_macro_f1':best_score}
    temp=fn.with_suffix('.partial.pt')
    torch.save(record,temp);os.replace(temp,fn)
    pd.DataFrame(history).to_csv(OUT/f'size_training_history_seed{seed}.csv',index=False)
    return model,record

models_by_seed={}; training_rows=[]
for seed in SEEDS:
    net,record=train_or_load(seed)
    models_by_seed[seed]=net
    training_rows.append({'seed':seed,'selected_epoch':record['best_epoch'],
                          'development_macro_f1':record['best_macro_f1']})
print('Development-only selected grading checkpoints:')
display(pd.DataFrame(training_rows))

## 6. One spatial anomaly detector, identical in both arms


In [ ]:
# Shared fixed projection, independent of all test images or labels.
rng_proj=np.random.default_rng(20261022)
Q,_=np.linalg.qr(rng_proj.standard_normal((512,PATCH_DIM)))
PROJECT=torch.from_numpy(Q.astype(np.float32)).to(DEVICE)

@torch.inference_mode()
def spatial_project(net,cache,batch=INFER_BATCH):
    net.eval()
    parts=[]
    for start in range(0,len(cache),batch):
        x=torch.from_numpy(np.asarray(cache[start:start+batch],dtype=np.float32)).to(DEVICE)
        fmap=net.spatial(x)
        must(fmap.shape[1:]==(512,7,7),f'Unexpected layer4 spatial shape {fmap.shape}')
        # B,C,H,W -> B,49,512 -> B,49,64
        flat=fmap.permute(0,2,3,1).reshape(len(x),49,512)
        part=torch.matmul(flat,PROJECT).detach().cpu().numpy().astype(np.float32)
        parts.append(part)
    return np.concatenate(parts,axis=0)

# Generic layer4 has exactly the same pretrained weights that seeded each supervised layer4.
generic=SizeHead(base.layer4).eval().to(DEVICE)
for p in generic.parameters():p.requires_grad_(False)

class SpatialNormalModel:
    def __init__(self):
        self.mu=None;self.whiten=None
    def fit(self,train):
        arr=np.asarray(train,dtype=np.float64)
        must(arr.ndim==3 and arr.shape[1:]==(49,PATCH_DIM),'Wrong patch training dimensions')
        must(np.isfinite(arr).all(),'Nonfinite normal patches')
        self.mu=arr.mean(axis=0).astype(np.float32)
        whitening=[]
        for p in range(49):
            centered=arr[:,p,:]-self.mu[p]
            covariance=centered.T@centered/len(centered)
            iso=float(np.trace(covariance))/PATCH_DIM
            cov=((1-COV_SHRINKAGE)*covariance+
                 (COV_SHRINKAGE*iso+1e-4)*np.eye(PATCH_DIM))
            chol=np.linalg.cholesky(cov)
            whitening.append(np.linalg.inv(chol.T).astype(np.float32))
        self.whiten=np.stack(whitening,axis=0)
        must(self.whiten.shape==(49,PATCH_DIM,PATCH_DIM),'Malformed Gaussian patch scorer')
        return self
    def score(self,variants,batch=128):
        must(self.mu is not None and self.whiten is not None,'Unfitted patch scorer')
        out=np.empty(len(variants),dtype=np.float32)
        for start in range(0,len(variants),batch):
            v=np.asarray(variants[start:start+batch],dtype=np.float32)
            centered=np.transpose(v-self.mu,(1,0,2))  # spatial_position,batch,dim
            white=np.matmul(centered,self.whiten)
            patch_dist=np.maximum(np.sum(white*white,axis=2).T,0.)
            out[start:start+len(v)]=np.partition(patch_dist,-TOP_PATCHES,axis=1)[:,-TOP_PATCHES:].mean(1)
        must(np.isfinite(out).all(),'Nonfinite patch anomaly scores')
        return out

# Deterministic unit test: abnormal single spatial location should score higher.
prng=np.random.default_rng(1929)
tr=prng.normal(size=(80,49,PATCH_DIM)).astype(np.float32)
probe_score=SpatialNormalModel().fit(tr)
control=tr[:6].copy();modified=control.copy();modified[0,9,:]+=10.0
s0=probe_score.score(control);s1=probe_score.score(modified)
must(s1[0]>s0[0],'Anomaly scorer cannot detect a large synthetic patch deviation in unit test')
print('PASS: fixed spatial Gaussian scoring unit test.')

# Generic patches/scores are calculated ONCE and shared across all three grader seeds.
generic_train=spatial_project(generic,fixed['train'])
generic_density=SpatialNormalModel().fit(generic_train)
generic_scores={p:generic_density.score(spatial_project(generic,fixed[p]))
                for p in ('calibration','locked_test')}
print('Generic patch scores:',{k:(len(v),float(v.min()),float(v.max())) for k,v in generic_scores.items()})
del generic_train

## 7. Fixed referral rules and the four-way grading decision


In [ ]:
def empirical_rank(clean_reference,values):
    ref=np.sort(np.asarray(clean_reference,dtype=np.float64))
    must(len(ref)==556,'Rank reference must contain 556 clean calibration scores')
    return np.searchsorted(ref,np.asarray(values,dtype=np.float64),side='right')/len(ref)

def cal_and_apply(calibration,testing,cal_normal_mask,budget):
    ref=np.asarray(calibration,dtype=np.float64)[cal_normal_mask]
    th=float(np.quantile(ref,1-budget,method='higher'))
    return th,np.asarray(testing,dtype=np.float64)>th

cal_clean=cal.kind.eq('original').to_numpy()
test_clean=test.kind.eq('original').to_numpy()
test_benign=test.kind.str.startswith('benign').to_numpy()
test_edit=test.label.eq('1').to_numpy()
must(sum(cal_clean)==556 and sum(test_clean)==834 and sum(test_edit)==10008,'Outcome masks inconsistent')
cal_truth=np.asarray([label_ids[z] for z in cal.source_size],dtype=np.int64)
test_truth=np.asarray([label_ids[z] for z in test.source_size],dtype=np.int64)

# Other than small numerical reproducibility tolerance, no task-dependent conditional choices.
metrics=[];family_metrics=[];predictions=[];paired_differences=[]
for seed in SEEDS:
    print(f'\n=== V5.1 matched evaluation, seed {seed} ===')
    net=models_by_seed[seed]
    grade_cal_logits=predict_logits(net,fixed['calibration'])
    grade_test_logits=predict_logits(net,fixed['locked_test'])
    grade_cal_prob=torch.softmax(torch.from_numpy(grade_cal_logits),dim=1).numpy()
    grade_test_prob=torch.softmax(torch.from_numpy(grade_test_logits),dim=1).numpy()
    cal_unc=1-grade_cal_prob.max(axis=1)
    test_unc=1-grade_test_prob.max(axis=1)
    y_cal_pred=grade_cal_prob.argmax(axis=1)
    y_test_pred=grade_test_prob.argmax(axis=1)
    clean_acc=float(accuracy_score(test_truth[test_clean],y_test_pred[test_clean]))
    development_pred=predict_logits(net,fixed['development']).argmax(axis=1)
    dev_acc=float(accuracy_score(Y_DEV,development_pred))
    print(f'Development size accuracy {dev_acc:.4f}; exploratory test clean accuracy {clean_acc:.4f}')

    supervised_train=spatial_project(net,fixed['train'])
    supervised_density=SpatialNormalModel().fit(supervised_train)
    supervised_scores={p:supervised_density.score(spatial_project(net,fixed[p]))
                       for p in ('calibration','locked_test')}
    del supervised_train
    # Guard against accidentally applying the same pretrained features in both arms.
    representation_shift=float(np.mean(np.abs(generic_scores['calibration']-supervised_scores['calibration'])))
    must(representation_shift>1e-5,'Size training did not change projected anomaly scores')

    streams={
       'confidence_only':(cal_unc,test_unc),
       'generic_spatial':(generic_scores['calibration'],generic_scores['locked_test']),
       'size_spatial':(supervised_scores['calibration'],supervised_scores['locked_test']),
       'joint_generic':(np.maximum(empirical_rank(cal_unc[cal_clean],cal_unc),
                                   empirical_rank(generic_scores['calibration'][cal_clean],generic_scores['calibration'])),
                        np.maximum(empirical_rank(cal_unc[cal_clean],test_unc),
                                   empirical_rank(generic_scores['calibration'][cal_clean],generic_scores['locked_test']))),
       'joint_size':(np.maximum(empirical_rank(cal_unc[cal_clean],cal_unc),
                                empirical_rank(supervised_scores['calibration'][cal_clean],supervised_scores['calibration'])),
                     np.maximum(empirical_rank(cal_unc[cal_clean],test_unc),
                                empirical_rank(supervised_scores['calibration'][cal_clean],supervised_scores['locked_test'])))
    }
    # Paired clean source predictions define edit-induced new mistakes.
    clean_rows=test.loc[test_clean,['source_id','source_group']].copy()
    clean_rows['clean_pred']=y_test_pred[test_clean]
    clean_rows['clean_correct']=y_test_pred[test_clean]==test_truth[test_clean]
    pair=test[['sample_id','source_id','source_group','family','severity','kind','source_variety','source_size']].merge(
        clean_rows[['source_id','clean_pred','clean_correct']],on='source_id',validate='many_to_one')
    pair['pred_size']=[label_order[i] for i in y_test_pred]
    pair['grade_error']=y_test_pred!=test_truth
    pair['new_edit_error']=test_edit & pair['clean_correct'].to_numpy() & (y_test_pred!=test_truth)
    pair['edit_grade_changed']=test_edit & (y_test_pred!=pair.clean_pred.to_numpy())
    pair['max_grade_probability']=grade_test_prob.max(1)
    for policy,(sc,st) in streams.items():
        pair[f'score_{policy}']=st
        for budget in BUDGETS:
            th,ref=cal_and_apply(sc,st,cal_clean,budget)
            pair[f'refer_{policy}_{int(round(budget*100))}']=ref
            keep=(~ref)&test_clean
            accepted_error=float(np.mean(pair.loc[keep,'grade_error'])) if keep.any() else np.nan
            err_clean=pair.grade_error.to_numpy()&test_clean
            new_error=pair.new_edit_error.to_numpy()
            changed=pair.edit_grade_changed.to_numpy()
            metrics.append({
                'seed':seed,'policy':policy,'budget':budget,'threshold':th,
                'development_grade_accuracy':dev_acc,'test_original_grade_accuracy':clean_acc,
                'clean_fpr':float(ref[test_clean].mean()),
                'benign_fpr':float(ref[test_benign].mean()),
                'synthetic_recall':float(ref[test_edit].mean()),
                'original_coverage':float(keep.sum()/test_clean.sum()),
                'accepted_original_grade_error':accepted_error,
                'clean_grade_errors_referred':int(np.count_nonzero(ref&err_clean)),
                'n_clean_errors':int(err_clean.sum()),
                'new_edit_error_recall':float(ref[new_error].mean()) if new_error.any() else np.nan,
                'n_new_edit_errors':int(new_error.sum()),
                'edited_changed_grade_recall':float(ref[changed].mean()) if changed.any() else np.nan,
                'n_changed_grade_edits':int(changed.sum()),
                'size_supervision_score_shift':representation_shift})
            if policy in ('generic_spatial','size_spatial','joint_generic','joint_size'):
                for (fam,severity),idx in test.loc[test_edit].groupby(['family','severity']).groups.items():
                    loc=np.asarray(list(idx),dtype=int)
                    family_metrics.append({'seed':seed,'policy':policy,'budget':budget,
                        'family':fam,'severity':severity,'n':len(loc),
                        'synthetic_recall':float(ref[loc].mean())})
    predictions.append(pair)
    print('Primary budget policies:')
    display(pd.DataFrame(metrics).query('seed == @seed and budget == @PRIMARY_BUDGET')[
        ['policy','clean_fpr','benign_fpr','synthetic_recall',
         'accepted_original_grade_error','new_edit_error_recall']].round(4))

main=pd.DataFrame(metrics)
family=pd.DataFrame(family_metrics)
preds=pd.concat([p.assign(seed=s) for s,p in zip(SEEDS,predictions)],ignore_index=True)
main.to_csv(OUT/'v51_main_policy_metrics.csv',index=False)
family.to_csv(OUT/'v51_family_severity.csv',index=False)
preds.to_csv(OUT/'v51_per_sample_predictions.csv',index=False)
pd.DataFrame(training_rows).to_csv(OUT/'v51_development_model_selection.csv',index=False)
print('Saved all five referral policies for three seeds × three prespecified budgets.')

## 8. Source-paired uncertainty and referral decisions


In [ ]:
def source_paired_ci(frame,flag_a,flag_b,mask,bseed):
    use=frame.loc[mask,['source_group',flag_a,flag_b]].copy()
    grouped=use.groupby('source_group',sort=True)[[flag_a,flag_b]].mean()
    if grouped.empty:return (np.nan,np.nan,np.nan)
    diffs=(grouped[flag_a]-grouped[flag_b]).to_numpy(dtype=np.float64)
    observed=float(diffs.mean())
    rng=np.random.default_rng(bseed)
    boots=np.empty(BOOT_REPS,dtype=np.float64)
    for b in range(BOOT_REPS):
        selected=rng.integers(0,len(diffs),size=len(diffs))
        boots[b]=float(diffs[selected].mean())
    low,high=np.quantile(boots,[0.025,0.975])
    return observed,float(low),float(high)

contrast_defs={
  'supervised_minus_generic_spatial':('size_spatial','generic_spatial'),
  'joint_size_minus_joint_generic':('joint_size','joint_generic'),
  'joint_size_minus_confidence':('joint_size','confidence_only')
}
result_rows=[]
for seed in SEEDS:
    one=preds.loc[preds.seed.eq(seed)].reset_index(drop=True)
    must(len(one)==12510 and one.source_group.nunique()==834,'Bootstrap population mismatch')
    must(one.label.eq('1').sum()==10008,'Synthetic bootstrap outcome mask must contain 10,008 known edited samples')
    for label,(a,b) in contrast_defs.items():
        for budget in BUDGETS:
            aa=f'refer_{a}_{int(round(budget*100))}'
            bb=f'refer_{b}_{int(round(budget*100))}'
            for outcome,mask in [('clean_false_referral',one.kind.eq('original').to_numpy()),
                                 ('benign_false_referral',one.kind.str.startswith('benign').to_numpy()),
                                 ('synthetic_referral',one.label.eq('1').to_numpy()),
                                 ('new_edit_error_referral',one.new_edit_error.to_numpy())]:
                delta,lo,hi=source_paired_ci(one,aa,bb,mask,20261022+seed+int(budget*100))
                result_rows.append({'seed':seed,'contrast':label,'budget':budget,'outcome':outcome,
                                    'delta':delta,'ci95_lower':lo,'ci95_upper':hi,
                                    'n_groups':int(one.loc[mask,'source_group'].nunique())})
paired=pd.DataFrame(result_rows)
paired.to_csv(OUT/'v51_source_group_bootstrap.csv',index=False)
print('Primary contrast at 10% calibration referral:')
display(paired.query('contrast == "supervised_minus_generic_spatial" and budget == @PRIMARY_BUDGET')
        [['seed','outcome','delta','ci95_lower','ci95_upper','n_groups']].round(4))

primary=main.loc[main.budget.eq(PRIMARY_BUDGET)].copy()
rows=[]
for seed in SEEDS:
    x=primary[primary.seed.eq(seed)].set_index('policy')
    generic=x.loc['generic_spatial'];supervised=x.loc['size_spatial']
    joint=x.loc['joint_size'];joint_generic=x.loc['joint_generic'];confidence=x.loc['confidence_only']
    joint_visual_value=bool(
        joint.synthetic_recall-confidence.synthetic_recall>=0.05 and
        joint.clean_fpr-confidence.clean_fpr<=0.01 and
        joint.benign_fpr-confidence.benign_fpr<=0.01)
    joint_grading_value=bool(
        confidence.accepted_original_grade_error-joint.accepted_original_grade_error>=0.01 and
        joint.clean_fpr-confidence.clean_fpr<=0.01)
    joint_incremental_generic=bool(
        (joint.synthetic_recall-joint_generic.synthetic_recall>=0.05 and
         joint.clean_fpr-joint_generic.clean_fpr<=0.01 and
         joint.benign_fpr-joint_generic.benign_fpr<=0.01) or
        (joint_generic.accepted_original_grade_error-joint.accepted_original_grade_error>=0.01 and
         joint.clean_fpr-joint_generic.clean_fpr<=0.01))
    rows.append({'seed':seed,
        'spatial_recall_improvement_pp':100*(supervised.synthetic_recall-generic.synthetic_recall),
        'clean_fpr_difference_pp':100*(supervised.clean_fpr-generic.clean_fpr),
        'benign_fpr_difference_pp':100*(supervised.benign_fpr-generic.benign_fpr),
        'joint_synthetic_advantage_vs_confidence_pp':100*(joint.synthetic_recall-confidence.synthetic_recall),
        'joint_accepted_grade_error_difference_pp':100*(joint.accepted_original_grade_error-
                                                        confidence.accepted_original_grade_error),
        'joint_visual_value_vs_confidence':joint_visual_value,
        'joint_grading_value_vs_confidence':joint_grading_value,
        'joint_incremental_over_generic_fusion':joint_incremental_generic,
        'joint_adds_value':bool(joint_visual_value or joint_grading_value),
        'improves_detection_at_least_5pp':bool((supervised.synthetic_recall-generic.synthetic_recall)>=0.05),
        'normal_false_referral_within_1pp':bool((supervised.clean_fpr-generic.clean_fpr)<=0.01),
        'benign_false_referral_within_1pp':bool((supervised.benign_fpr-generic.benign_fpr)<=0.01)})
gate=pd.DataFrame(rows)
gate.to_csv(OUT/'v51_fixed_go_stop_criteria.csv',index=False)
display(gate.round(3))
meets=bool(gate[['improves_detection_at_least_5pp','normal_false_referral_within_1pp',
                 'benign_false_referral_within_1pp','joint_adds_value']].all().all())
if not meets:
    print('PREDEFINED STOP: Size-supervised anomaly detection did not satisfy the operational gate '
          'consistently across all three seeds. Do not launch another detector search.')
else:
    print('PREDEFINED GATE SATISFIED on legacy exploratory data only. '
          'Independent sealed confirmation still required; this does not establish real-defect performance.')

## 9. Evaluation outputs and limitations


In [ ]:
import matplotlib.pyplot as plt

fig,axes=plt.subplots(1,2,figsize=(13,4.7))
show=primary.groupby('policy')[['synthetic_recall','clean_fpr','accepted_original_grade_error']].mean()
names=['confidence_only','generic_spatial','size_spatial','joint_generic','joint_size']
xx=np.arange(len(names))
axes[0].bar(xx,100*show.loc[names,'synthetic_recall'].to_numpy())
axes[0].set_xticks(xx,names,rotation=25,ha='right')
axes[0].set_title('Held-out synthetic-edit referral: 10% calibration target')
axes[0].set_ylabel('Synthetic edit recall (%)')
axes[1].bar(xx-0.18,100*show.loc[names,'clean_fpr'].to_numpy(),width=0.36,label='Clean referral')
axes[1].bar(xx+0.18,100*show.loc[names,'accepted_original_grade_error'].to_numpy(),
            width=0.36,label='Accepted clean grade error')
axes[1].set_xticks(xx,names,rotation=25,ha='right')
axes[1].set_title('Clean referral and retained grading error')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend()
fig.suptitle('V5.1 exploratory matched spatial features — all three seeds')
fig.tight_layout()
fig.savefig(OUT/'v51_spatial_primary_comparison.png',dpi=175,bbox_inches='tight')
plt.show()

mixed=src.loc[src.split.eq('train')].groupby('source_group').native_size_label.nunique()
review={'multi_size_training_source_groups':int(mixed.gt(1).sum()),
        'images_in_multi_size_training_source_groups':int(src.loc[(src.split=='train') &
                  src.source_group.isin(mixed[mixed.gt(1)].index)].shape[0])}
atomic_json(OUT/'training_group_sensitivity_inventory.json',review)


# Predeclared reporting: variety and size breakdowns for the grade-or-refer system.
size_by_variety=[];grade_confusion=[]
for seed in SEEDS:
    originals=preds.loc[preds.seed.eq(seed) & preds.kind.eq('original')].copy()
    must(len(originals)==834,'Missing original grading evaluations for variety breakdown')
    for (variety,size),g in originals.groupby(['source_variety','source_size']):
        size_by_variety.append({'seed':seed,'variety':variety,'true_size':size,
             'n':len(g),'grading_accuracy':float((g.pred_size==g.source_size).mean())})
    for true_size in label_order:
        for predicted_size in label_order:
            grade_confusion.append({'seed':seed,'true_size':true_size,'predicted_size':predicted_size,
                'n':int(((originals.source_size==true_size)&(originals.pred_size==predicted_size)).sum())})
pd.DataFrame(size_by_variety).to_csv(OUT/'v51_grading_by_variety_size.csv',index=False)
pd.DataFrame(grade_confusion).to_csv(OUT/'v51_grading_confusion.csv',index=False)
print('Saved variety/size diagnostics and full three-seed grading confusion counts.')

required_files=['v51_main_policy_metrics.csv','v51_family_severity.csv','v51_per_sample_predictions.csv',
                'v51_source_group_bootstrap.csv','v51_fixed_go_stop_criteria.csv',
                'v51_development_model_selection.csv','v51_spatial_primary_comparison.png',
                'v51_grading_by_variety_size.csv','v51_grading_confusion.csv',
                'training_group_sensitivity_inventory.json','provenance.json','weights_provenance.json']
must(all((OUT/x).is_file() and (OUT/x).stat().st_size>0 for x in required_files),
     'Some final outputs were not created')
must(len(main)==len(SEEDS)*5*len(BUDGETS) and len(preds)==len(SEEDS)*len(test),
     'Final result table row count is incorrect')
atomic_json(OUT/'RESULTS_COMPLETE.json',{'revision':REVISION,'finished_utc':pd.Timestamp.now(tz='UTC').isoformat(),
    'success_gate_met_on_exploratory_legacy_test':meets,'artifacts':required_files,
    'test_is_confirmatory':False,'real_defect_labels_used_in_training':False,
    'original_V4_mutated':False,'scope':'One matched spatial representation experiment, three seeds.'})
print('EXPERIMENT COMPLETED. Results:',OUT)
print('V4 was read-only. No V4 detector was trained. No synthetic benchmark was regenerated.')
print('Use the locked protocol stopping criteria. If negative, close the hypothesis rather than grow another benchmark.')